# Homework 2: BERT Finetuning for Robust Phrasal Chunking

## 1. Introduction

Phrasal chunking is the task of identifying non-recursive syntactic groups (chunks) in a sentence, such as noun phrases (NP), verb phrases (VP), and prepositional phrases (PP).

The key challenge in this assignment is robustness to noisy input. The training data contains clean text from the [CoNLL-2000 shared task](https://arxiv.org/abs/cs/0009008), the `dev.txt` and `test.txt` sets contain intentional spelling errors and typos (e.g., `Rockwell` becomes `Rqckwell`). This simulates real-world scenarios where NLP models must handle misspelled or corrupted text.

Our baseline is a [DistilBERT](https://arxiv.org/abs/1910.01108) model fine-tuned with a single linear classification head. We improved this baseline through four major modifications:

1. **Data augmentation with noise injection** to close the train-test distribution gap
2. **A multi-layer perceptron (MLP) classification head** for more complex feature transformation & non-linear activations
3. **Differential learning rates** with separate optimizers for the encoder and classification head
4. **Multi-layer BERT representations** by averaging the last four hidden layers to take advantage of syntax related features in the middle layers.

## 2. Method

### 2.1 Baseline Architecture

The baseline fine-tunes [DistilBERT](https://arxiv.org/abs/1910.01108) (`distilbert-base-uncased`) as a sequence tagger. Input words are tokenized into WordPiece subwords, each mapped to a 768-dimensional contextualized representation by the encoder's last hidden layer. A single linear layer projects these to the tag space, followed by log-softmax. For multi-subword words, the first subword's prediction is used. Training uses Adam (lr=5e-5), batch size 4, NLLLoss, and 5 epochs on the clean CoNLL-2000 data (8,936 sentences).

### 2.2 Improvement 1: Data Augmentation with Noise Injection

The most critical challenge is the mismatch between clean training data and noisy dev/test data. Inspired by [adversarial training with misspellings](https://aclanthology.org/P19-1561/), we apply **character-level noise injection** to create augmented training examples. For each epoch, we add a noisy copy of every training sentence alongside the original clean version, doubling the effective training size to 17,872 sentences.

For each word in a sentence (length > 2), a probability of 0.2 is applied to corrupt the word. The corruption types are as follows:

| Type | Operation | Example |
|:---|:---|:---|
| **Swap** | Transpose two adjacent characters | "expected" → "expceted" |
| **Delete** | Remove a random character | "Confidence" → "Confdence" |
| **Insert** | Insert a random letter at a random position | "tomorrow" → "xtomorrow" |
| **Substitute** | Replace a random character with a random letter | "improvement" → "improeement" |

Below are two sample augmented sentences, with corrupted words highlighted:

> **Original:**
> 
> Confidence in the pound is widely expected to take another sharp dive if trade figures for September , due for release tomorrow , fail to show a substantial improvement from July and August 's near-record deficits .
>
> **Noisy:**
> 
> **Confdence** in the pound is widely **expecte** to take another sharp dive if trade figures **fon** September , due **ofr** release **xtomorrow** , **afil** to **sohw** a substantial **improeement** **fro** **Juliy** **band** **uAgust** 's near-record deficits .

> **Original:** 
> 
> Chancellor of the Exchequer Nigel Lawson 's restated commitment to a firm monetary policy has helped to prevent a freefall in sterling over the past week .
>
> **Noisy:**
> 
> Chancellor of the **Exchquer** **Nigtel** Lawson 's restated **commitmnet** to a **fimr** **moonetary** policy has helped to **prevnt** a freefall in sterling over **thm** past **wee** .

The augmented data is reshuffled each epoch so the model sees different noisy variants across epochs, acting as a form of regularization. Words of length ≤ 2 are never corrupted, preserving function words and punctuation.

### 2.3 Improvement 2: MLP Classification Head

The baseline uses a single linear layer to project encoder representations to tag space. We replace this with a **two-layer MLP** with ReLU activations and dropout:

1. **Dropout** (p=0.1) on the encoder output for regularization
2. **Linear layer 1**: 768 → 3072 (4× expansion)
3. **ReLU activation**
4. **Linear layer 2**: 3072 → 768 (projection back)
5. **ReLU activation**
6. **Classification linear layer**: 768 → number of tags

The wider intermediate layer (4× the hidden dimension) allows the model to learn more complex, nonlinear transformations of the BERT representations before classification. The dropout layer helps prevent overfitting, especially when we have a large amount of augmented data in training.

### 2.4 Improvement 3: Differential Learning Rates

Fine-tuning pretrained language models benefits from using different learning rates for the pretrained encoder and the randomly initialized classification head. The pretrained encoder weights already encode useful linguistic knowledge and should be updated conservatively, while the new classification layers need to be trained more aggressively from scratch.

We use two separate optimizers and experiment with several variants:
- **Dual Adam**: Adam (lr=5e-5) for the encoder, Adam (lr=1e-3) for the head
- **Adam + SGD (lr=0.1)**: Adam (lr=5e-5) for the encoder, SGD (lr=0.1, momentum=0.9) for the head
- **Adam + SGD (lr=0.01)**: Adam (lr=5e-5) for the encoder, SGD (lr=0.01, momentum=0.9) for the head

### 2.5 Improvement 4: Multi-Layer BERT Representations

Different layers of pretrained language models capture different types of linguistic information: lower layers tend to capture surface-level and syntactic features, while upper layers capture more semantic information. Rather than using only the last hidden layer, we extract the **last four hidden layers** from DistilBERT and **average** them element-wise to produce the final token representations.

This is achieved by passing `output_hidden_states=True` to the encoder and stacking the last four hidden state tensors along a new dimension before averaging.

## 3. Experiments and Results

### 3.1 Experimental Setup

All experiments are evaluated on `data/input/dev.txt` using the CoNLL evaluation script (`conlleval.py`). The script reports overall accuracy (non-O) and F1 score at the phrase level, as well as per-chunk-type breakdowns.

We added one improvement at a time starting from the baseline. Experiments branch at the optimizer choice (Exp3 vs Exp4/Exp8) and again when multi-layer representations are added (Exp5, Exp6, Exp7), allowing us to compare different optimizer–representation combinations.


### 3.2 Overall Results

| Exp | Noise Augmentation | MLP Head | Optimizer | Multi-layer Features | F1 (%) | ΔF1 |
|:---:|:---:|:---:|:---|:---:|:---:|:---:|
| Baseline | ✗ | ✗ | Single Adam | ✗ | 90.70 | — |
| Exp1 | ✓ | ✗ | Single Adam | ✗ | 94.49 | +3.79 |
| Exp2 | ✓ | ✓ | Single Adam | ✗ | 94.41 | +3.71 |
| Exp3 | ✓ | ✓ | Dual Adam | ✗ | 94.67 | +3.97 |
| Exp4 | ✓ | ✓ | Adam + SGD (lr=0.1) | ✗ | 94.52 | +3.82 |
| Exp5 | ✓ | ✓ | Adam + SGD (lr=0.1) | ✓ | 94.21 | +3.51 |
| Exp6 | ✓ | ✓ | Dual Adam | ✓ | 94.64 | +3.94 |
| **Exp7** | **✓** | **✓** | **Adam + SGD (lr=0.01)** | **✓** | **94.81** | **+4.11** |
| Exp8 | ✓ | ✓ | Adam + SGD (lr=0.01) | ✗ | 94.50 | +3.80 |

- **Single Adam**: one Adam optimizer (lr=5e-5) for all parameters.
- **Dual Adam**: Adam (lr=5e-5) for encoder + Adam (lr=1e-3) for classification head.
- **Adam + SGD (lr=X)**: Adam (lr=5e-5) for encoder + SGD (lr=X, momentum=0.9) for classification head.


### 3.3 Per-Chunk-Type F1 Results

| Exp | ADJP | ADVP | CONJP | NP | PP | PRT | SBAR | VP |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|:---:|
| Baseline | 73.73 | 73.82 | 80.00 | 91.19 | 95.46 | 74.70 | 83.18 | 90.03 |
| Exp1 | 77.88 | 79.84 | 58.82 | 95.00 | 97.84 | 75.79 | 89.76 | 94.60 |
| Exp2 | 75.80 | 78.02 | 0.00 | 94.95 | 97.66 | 71.91 | 90.48 | 95.16 |
| Exp3 | 77.50 | 79.53 | 82.35 | 95.08 | 98.11 | 75.51 | 89.09 | 95.17 |
| Exp4 | 75.74 | 78.50 | 50.00 | 95.24 | 97.87 | 75.51 | 89.38 | 94.63 |
| Exp5 | 76.21 | 77.84 | 66.67 | 94.64 | 97.80 | 72.34 | 89.47 | 94.84 |
| Exp6 | 78.60 | 79.16 | 73.68 | 95.21 | 98.00 | 73.68 | 86.94 | 95.06 |
| **Exp7** | **77.52** | **79.95** | **33.33** | **95.42** | **97.95** | **77.89** | **89.62** | **95.21** |
| Exp8 | 77.16 | 79.79 | 83.33 | 95.02 | 97.75 | 71.58 | 89.09 | 94.82 |

### 3.4 Ablation Analysis

#### Effect of Noise Augmentation (Baseline → Exp1): +3.79 F1

Noise augmentation is by far the single most impactful improvement, increasing F1 from 90.70 to 94.49. The increase reflected in many chunktypes: NP improves by +3.81, VP by +4.57, PP by +2.38, and SBAR by +6.58. Token errors drop from 1300 to 773 (a 40.5% reduction). This confirms that the train-test distribution mismatch due to noisy input was the dominant source of errors, and adversarial training improved robustness.

Note that the baseline's top error type was `B-NP → I-NP` (228 times), indicating the model frequently failed to detect NP boundaries, probably because misspelled words produced unfamiliar subword tokenizations. After augmentation, this error drops to 68 occurrences.

#### Effect of MLP Classification Head (Exp1 vs Exp2/Exp3/Exp4/Exp8):

To isolate the effect of the MLP head, we compare Exp1 (linear head) against all experiments that add the MLP head with different optimizer configurations:

| Comparison | Head | Optimizer | F1 (%) | ΔF1 vs Exp1 |
|:---|:---|:---|:---:|:---:|
| Exp1 | Linear | Single Adam | 94.49 | — |
| Exp2 | MLP | Single Adam | 94.41 | −0.08 |
| Exp3 | MLP | Dual Adam | 94.67 | +0.18 |
| Exp4 | MLP | Adam + SGD (lr=0.1) | 94.52 | +0.03 |
| Exp8 | MLP | Adam + SGD (lr=0.01) | 94.50 | +0.01 |

With a single shared Adam optimizer (Exp2), the MLP head slightly decrease performance (−0.08), indicating that a single optimizer with lr=5e-5 is too conservative for the new multi-layer classification head.

However, if implemented together wiht dual optimizers, the MLP head becomes beneficial to the overall performance. Dual Adam (Exp3) achieves the best gain (+0.18 over Exp1). The Adam+SGD variants (Exp4, Exp8) show smaller improvements, suggesting that Adam's adaptive per-parameter learning rates are better suited for the MLP head than SGD's fixed rate.

This demonstrates that the MLP head and differential learning rates are tightly coupled improvements.

#### Effect of Differential Learning Rates (Exp2 → Exp3/Exp4/Exp8):

Applying different learning rate for BERT encoders and classification head showed clear increase in the performance:
- **Dual Adam** (Exp3): 94.41 → 94.67 (+0.26). CONJP recovers to 82.35, PP reaches 98.11.
- **Adam + SGD lr=0.1** (Exp4): 94.41 → 94.52 (+0.11). Moderate improvement; NP is best at 95.24 among non-multi-layer runs.
- **Adam + SGD lr=0.01** (Exp8): 94.41 → 94.50 (+0.09). Similar to Exp4; CONJP jumps to 83.33.

Dual Adam (Exp3) outperforms both SGD variants without multi-layer representations. The Adam optimizer's adaptive learning rates appear to help the classification head converge more effectively than SGD at this stage.

#### Effect of Multi-Layer Representations (Exp4 → Exp5, Exp3 → Exp6, Exp8 → Exp7):

The effect of multi-layer averaging depends heavily on the optimizer:
- **Adam + SGD lr=0.1** (Exp4 → Exp5): 94.52 → 94.21 (−0.31). Multi-layer *hurts* with a high SGD learning rate. The aggressive SGD updates may destabilize the averaged representation.
- **Dual Adam** (Exp3 → Exp6): 94.67 → 94.64 (−0.03). Essentially neutral.
- **Adam + SGD lr=0.01** (Exp8 → Exp7): 94.50 → **94.81** (+0.31). The best overall result. The conservative SGD learning rate combined with multi-layer representations produces the best combination.

This shows that multi-layer representations benefit from a conservative head learning rate, letting the model to learn meaningful combinations of syntactic and semantic features.

#### Summary of incremental contributions:
| Transition | Change | ΔF1 |
|:---|:---|:---:|
| Baseline → Exp1 | + Noise augmentation | +3.79 |
| Exp1 → Exp2 | + MLP head | −0.08 |
| Exp2 → Exp8 | + Adam + SGD (lr=0.01) | +0.09 |
| Exp8 → Exp7 | + Multi-layer repr. | +0.31 |
| **Total (Baseline → Exp7)** | **All improvements** | **+4.11** |

### 3.5 Qualitative Analysis

**Example 1: Noise augmentation fixes misspelling-induced errors (Sentence 1)**

> Rockwell **sabid** the agreement **clals** for it to supply 200 additional so-called shipsets for the planes .

| Word | Gold | Baseline | Exp7 (Best) |
|:---|:---:|:---:|:---:|
| Rockwell | B-NP | **I-NP** ✗ | B-NP ✓ |
| sabid | B-VP | B-VP ✓ | B-VP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ |
| agreement | I-NP | I-NP ✓ | I-NP ✓ |
| clals | B-VP | **B-NP** ✗ | B-VP ✓ |
| for | B-SBAR | B-SBAR ✓ | B-SBAR ✓ |
| it | B-NP | B-NP ✓ | B-NP ✓ |
| to | B-VP | B-VP ✓ | B-VP ✓ |
| supply | I-VP | I-VP ✓ | I-VP ✓ |
| 200 | B-NP | B-NP ✓ | B-NP ✓ |
| additional | I-NP | I-NP ✓ | I-NP ✓ |
| so-called | I-NP | I-NP ✓ | I-NP ✓ |
| shipsets | I-NP | I-NP ✓ | I-NP ✓ |
| for | B-PP | B-PP ✓ | B-PP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ |
| planes | I-NP | I-NP ✓ | I-NP ✓ |
| . | O | O ✓ | O ✓ |

The baseline makes two errors on this sentence. First, it tags "Rockwell" as `I-NP` instead of `B-NP`, failing to recognize it as the start of a new noun phrase maybe because "Rockwell" at sentence start followed by the misspelled "sabid" creates an unfamiliar context. Second, it misclassifies the misspelled verb "clals" (calls) as `B-NP`, unable to recognize its role as a verb. Exp7 correctly tags both, demonstrating that noise augmentation helps the model generalize better for noisy input.

**Example 2: Architectural improvements fix chunk boundaries (Sentence 38)**

> Of the combined \$ 114.4 million the two men were scheduled to reap **unlder** the buy-out , they agreed to invest in the buy-out just \$ 15 **millinn** , angering many of the thousands of workers asked **to** make pay **concessions** **so** the buy-out would be a success .

| Word | Gold | Baseline | Exp1 | Exp7 (Best) |
|:---|:---:|:---:|:---:|:---:|
| Of | B-PP | B-PP ✓ | B-PP ✓ | B-PP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| combined | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| $ | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| 114.4 | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| million | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| two | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| men | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| were | B-VP | B-VP ✓ | B-VP ✓ | B-VP ✓ |
| scheduled | I-VP | I-VP ✓ | I-VP ✓ | I-VP ✓ |
| to | I-VP | I-VP ✓ | I-VP ✓ | I-VP ✓ |
| reap | I-VP | **B-VP** ✗ | I-VP ✓ | I-VP ✓ |
| unlder | B-PP | **B-VP** ✗ | B-PP ✓ | B-PP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| buy-out | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| , | O | O ✓ | O ✓ | O ✓ |
| they | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| agreed | B-VP | B-VP ✓ | B-VP ✓ | B-VP ✓ |
| to | I-VP | I-VP ✓ | I-VP ✓ | I-VP ✓ |
| invest | I-VP | I-VP ✓ | I-VP ✓ | I-VP ✓ |
| in | B-PP | B-PP ✓ | B-PP ✓ | B-PP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| buy-out | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| just | B-NP | **I-NP** ✗ | B-NP ✓ | B-NP ✓ |
| $ | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| 15 | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| millinn | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| , | O | O ✓ | O ✓ | O ✓ |
| angering | B-VP | B-VP ✓ | B-VP ✓ | B-VP ✓ |
| many | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| of | B-PP | B-PP ✓ | B-PP ✓ | B-PP ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| thousands | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| of | B-PP | B-PP ✓ | B-PP ✓ | B-PP ✓ |
| workers | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| asked | B-VP | B-VP ✓ | B-VP ✓ | B-VP ✓ |
| to | B-VP | B-VP ✓ | **I-VP** ✗ | B-VP ✓ |
| make | I-VP | I-VP ✓ | I-VP ✓ | I-VP ✓ |
| pay | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| concessions | I-NP | I-NP ✓ | **B-NP** ✗ | I-NP ✓ |
| so | B-SBAR | **O** ✗ | B-SBAR ✓ | B-SBAR ✓ |
| the | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| buy-out | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| would | B-VP | B-VP ✓ | B-VP ✓ | B-VP ✓ |
| be | I-VP | I-VP ✓ | I-VP ✓ | I-VP ✓ |
| a | B-NP | B-NP ✓ | B-NP ✓ | B-NP ✓ |
| success | I-NP | I-NP ✓ | I-NP ✓ | I-NP ✓ |
| . | O | O ✓ | O ✓ | O ✓ |

This long sentence (49 tokens) shows progressive improvement across models. We highlight the phrase-level chunking differences around the error regions:

**Error region 1** (around "reap unlder"): The baseline fails to recognize the misspelled preposition "unlder" (under) and merges it into the preceding VP:
- Gold: ... [VP were scheduled to reap] [PP unlder] [NP the buy-out] ...
- Baseline: ... [VP were scheduled to] [VP reap unlder] [NP the buy-out] ... ✗
- Exp1 / Exp7: ... [VP were scheduled to reap] [PP unlder] [NP the buy-out] ... ✓

**Error region 2** (around "just $ 15 millinn"): The baseline merges "just" into the preceding NP instead of recognizing it as the start of a new NP:
- Gold: ... [NP the buy-out] [NP just $ 15 millinn] ...
- Baseline: ... [NP the buy-out just $ 15 millinn] ... ✗
- Exp1 / Exp7: ... [NP the buy-out] [NP just $ 15 millinn] ... ✓

**Error region 3** (around "to make pay concessions so"): Exp1 fixes the baseline's misspelling errors but introduces two new boundary errors, it merges "to" into the preceding VP "asked" and splits "pay concessions" into two NPs:
- Gold: ... [VP asked] [VP to make] [NP pay concessions] [SBAR so] ...
- Baseline: ... [VP asked] [VP to make] [NP pay concessions] so ... ✗ (misses SBAR)
- Exp1: ... [VP asked to make] [NP pay] [NP concessions] [SBAR so] ... ✗ (merges VPs, splits NP)
- Exp7: ... [VP asked] [VP to make] [NP pay concessions] [SBAR so] ... ✓

Exp7's architectural improvements (MLP head, differential LR, multi-layer representations) help it correctly resolve the VP boundary at "to" and keep "pay concessions" as a single NP which is a complex boundary decisions that require understanding the syntactic structure.

## 4. Conclusion

We improved the DistilBERT baseline through four complementary techniques: character-level noise augmentation, an MLP classification head, differential learning rates for encoder and head, and multi-layer representation averaging. The baseline achieves 90.70 F1 on the noisy dev set; our best model (Exp7) achieves 94.81 F1, an improvement of +4.11 points.

The most impactful single improvement was **noise augmentation** (+3.79 F1), which directly addresses the train-test distribution mismatch. The remaining improvements contribute more modestly but reveal an important finding: **multi-layer representations and optimizer choice interact significantly**.



## 5. References

- Sang, E. F., & Buchholz, S. (2000). Introduction to the CoNLL-2000 shared task: Chunking. arXiv preprint cs/0009008.
- Pruthi, D., Dhingra, B., & Lipton, Z. C. (2019). Combating adversarial misspellings with robust word recognition. arXiv preprint arXiv:1905.11268.
- Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. arXiv preprint arXiv:1910.01108.